## **GITHUB repository**
Link: https://github.com/HABOZ1/Deliverable_2.git

In [3]:
from pathlib import Path
import re
import json
import pandas as pd
import numpy as np

## **Task 1**

In [ ]:
def file_reader(file_path: Path) -> tuple[str, str]:
    """
    Read a file and return its content along with the encoding used.

    Returns:
        A tuple containing the file content (str) and the encoding used (str).
    """

    try:
        with file_path.open("r", encoding="utf-8") as f:
            file_text_content = f.read()
            encoding_used = "utf-8"
        return (file_text_content, encoding_used)

    except UnicodeDecodeError:
        with file_path.open("r", encoding="latin-1") as f:
            file_text_content = f.read()
            encoding_used = "latin-1"
        return (file_text_content, encoding_used)

def log_parser(log_file_path: Path) -> list[dict[str, str | float]]:
    """
    Parse a log file and extract valid transactions.
    
    Returns:
        A list of dictionaries, each representing a valid transaction with keys:
        - timestamp (str)
        - register_id (str)
        - item_id (str)
        - price (float)
        - currency (str)
        - status (str)
    """

    file_content, encoding_used = file_reader(log_file_path)

    valid_transactions = []

    for line in file_content.splitlines():
        if line.startswith("---") or line.startswith("===") or line.startswith("[DEBUG]"):
            continue

        else:
            timestamp = str(re.search(r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]", line).group(1))
            register_id = str(re.search(r"POS_(\d+)", line).group(0))
            item_id = str(re.search(r"item=(\w+)", line).group(1))
            price = float(re.search(r"price=(\d+\.\d+)", line).group(1))
            currency = str(re.search(r"price=\d+\.\d+\s+(\w+)", line).group(1))
            status = str(re.search(r"\((SUCCESS|REFUND|CANCELLED)\)", line).group(1))

            valid_transactions.append({
                "timestamp": timestamp,
                "register_id": register_id,
                "item_id": item_id,
                "price": price,
                "currency": currency,
                "status": status
            })

    return valid_transactions



('[2026-10-15 07:01:20] [POS_01] item=Croissant price=3.38 USD (CANCELLED)\n[2026-10-15 07:02:28] [POS_01] item=Latte price=4.61 USD (CANCELLED)\n[2026-10-15 07:04:35] [POS_01] item=Cappuccino price=4.69 USD (CANCELLED)\n[2026-10-15 07:04:55] [POS_01] item=Espresso price=2.83 USD (REFUND)\n=== NETWORK SYNC OK ===\n[2026-10-15 07:05:07] [POS_01] item=Cappuccino price=4.32 NOK (SUCCESS)\n[2026-10-15 07:06:22] [POS_01] item=Muffin price=3.02 EUR (SUCCESS)\n[2026-10-15 07:08:06] [POS_01] item=Muffin price=3.46 USD (SUCCESS)\n[2026-10-15 07:09:36] [POS_01] item=Cappuccino price=4.53 USD (SUCCESS)\n[2026-10-15 07:10:24] [POS_01] item=Espresso price=2.60 NOK (SUCCESS)\n[2026-10-15 07:11:53] [POS_01] item=Muffin price=3.95 EUR (SUCCESS)\n[2026-10-15 07:13:09] [POS_01] item=Muffin price=3.48 USD (SUCCESS)\n[2026-10-15 07:15:26] [POS_01] item=Latte price=4.37 NOK (REFUND)\n[2026-10-15 07:15:37] [POS_01] item=Muffin price=3.35 NOK (SUCCESS)\n[2026-10-15 07:16:47] [POS_01] item=Espresso price=2.69

## **Task 2**

In [5]:
def find_log_files(folder: Path) -> list[Path]:
    """
    Find all .log files in the specified directory and its subdirectories.

    Returns:
        A list of Path objects representing the found .log files.
    """
    log_files = []

    for path in folder.iterdir():
        if path.is_file() and path.suffix == ".log":
            log_files.append(path)

        elif path.is_dir():
            log_files.extend(find_log_files(path))

    return log_files


def find_active_registers(data: Path | dict | list) -> set[str]:
    """
    Find all active registers in a given JSON data structure.

    Returns:
        A set of strings representing the IDs of active registers.
    """
    registers = set()

    # Check if the input is a Path object, and if so, read the JSON data from the file
    if isinstance(data, Path):
        with open(data, "r") as file:
            data = json.load(file)

    # Check if the input is a dictionary, and if so, look for "id" and "active" keys
    if isinstance(data, dict):
        if "id" in data and "active" in data:
            if data["active"]:
                registers.add(data["id"])
        for value in data.values():
            registers.update(find_active_registers(value))

    # Check if the input is a list, and if so, check each item in the list for a dictionary for "active" and "id" keys
    elif isinstance(data, list):
        for item in data:
            registers.update(find_active_registers(item))

    return registers


print(find_log_files(Path("../Deliverable_2/data/")))
print(find_active_registers(Path("../Deliverable_2/data/store_config.json")))


[WindowsPath('../Deliverable_2/data/2026/library_cafe/POS_01/register_sales.log'), WindowsPath('../Deliverable_2/data/2026/library_cafe/POS_02/register_sales.log'), WindowsPath('../Deliverable_2/data/2026/library_cafe/POS_03/register_sales.log'), WindowsPath('../Deliverable_2/data/2026/science_building/POS_01/register_sales.log'), WindowsPath('../Deliverable_2/data/2026/science_building/POS_02/register_sales.log'), WindowsPath('../Deliverable_2/data/2026/science_building/POS_03/register_sales.log'), WindowsPath('../Deliverable_2/data/2026/student_union/POS_01/register_sales.log'), WindowsPath('../Deliverable_2/data/2026/student_union/POS_02/register_sales.log'), WindowsPath('../Deliverable_2/data/2026/student_union/POS_03/register_sales.log')]
{'POS_03', 'POS_02', 'POS_01'}


## **Task 3**

In [ ]:
def remove_price_outliers(log_file_path: Path) -> np.ndarray:
    """
    Remove outliers from price data in a log file.

    Returns:
        A NumPy array of cleaned price data with outliers replaced by the median of non-outlier prices.
    """

    price_data = [valid_transaction["price"] for valid_transaction in log_parser(log_file_path)]
    price_data = np.array(price_data, dtype=float).flatten()

    mean = np.mean(price_data)
    std_dev = np.std(price_data)
    outliers = np.abs(price_data - mean) > 3 * std_dev
    median = np.median(price_data[~outliers])

    price_data[outliers] = median

    return price_data

def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """
    Computes the element-wise refund ratio using NumPy array operations.

    Returns:
        A Numpy Array with the computed refund ratios.
    """

    refund_ratio = np.array(refund_count / (sales_count + 10 ** -6))

    return refund_ratio


## **Task 4**

In [ ]:
def create_dataframe_from_logs(transactions: list[dict[str, str | float]]) -> pd.DataFrame:
    """
    Create a DataFrame from a list of transaction logs, and set the "timestamp" column as the index.

    Returns:
        A pandas DataFrame containing the transaction data, and the "timestamp" column is converted to datetime and set as the index.
    """

    # Create a DataFrame from the list of transactions
    df = pd.DataFrame(transactions)

    # Convert the "timestamp" column to datetime and set it as the index
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp").sort_index()

    return df


def group_by_item_and_hour(df: pd.DataFrame) -> pd.DataFrame:
    """
    Group the DataFrame by "item_id", resample by hour and summing the "price" column.
    """
    
    return df.groupby("item_id").resample("1h")["price"].sum()


def create_revenue_pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a pivot table showing total revenue for each item per hour.
    """

    pivot_table = pd.pivot_table(
        df,
        values="price",
        index="item_id",
        columns="status",
        aggfunc="sum", # aggregate function to sum the prices for each item and status
        fill_value=0 # fill_value to replace NaN values with 0
    )

    return pivot_table


def calculate_refund_ratio_per_item(df: pd.DataFrame) -> dict[str, float]:
    """
    Calculate the refund ratio for each item in the DataFrame.

    Returns:
        A dictionary where keys are item IDs and values are the corresponding refund ratios rounded to three decimals.
    """

    refund_ratios = {}

    # Loop through each unique item_id in the DataFrame
    for item in df["item_id"].unique():

        # Filter the DataFrame for the current item_id
        item_df = df[df["item_id"] == item]

        # Calculate the number of successful sales and refunds for the current item
        sales_count = len(item_df[item_df["status"] == "SUCCESS"])
        refund_count = len(item_df[item_df["status"] == "REFUND"])

        # Compute the refund ratio and round it to three decimals
        refund_ratios[item] = round(float(compute_refund_ratio(sales_count, refund_count)), 3)

    return refund_ratios


test = create_dataframe_from_logs(log_parser("../Deliverable_2/Data/2026/library_cafe/POS_01/register_sales.log"))

#print(test.head())

print(group_by_item_and_hour(test))
print(create_revenue_pivot_table(test))
print(calculate_refund_ratio_per_item(test))


item_id     timestamp          
Cappuccino  2026-10-15 07:00:00      24.53
            2026-10-15 08:00:00    1011.79
            2026-10-15 09:00:00       4.01
Croissant   2026-10-15 07:00:00      19.12
            2026-10-15 08:00:00      33.40
Espresso    2026-10-15 07:00:00       5.72
            2026-10-15 08:00:00      18.72
            2026-10-15 09:00:00       3.06
Latte       2026-10-15 07:00:00      53.05
            2026-10-15 08:00:00      18.08
            2026-10-15 09:00:00       4.06
Muffin      2026-10-15 07:00:00      14.08
            2026-10-15 08:00:00      41.75
            2026-10-15 09:00:00       3.96
Sandwich    2026-10-15 07:00:00      37.59
            2026-10-15 08:00:00      50.93
Name: price, dtype: float64
status      CANCELLED   REFUND  SUCCESS
item_id                                
Cappuccino       8.31  1003.69    28.33
Croissant       18.20     7.31    27.01
Espresso         0.00    11.14    16.36
Latte            4.93    13.32    56.94
Muffin      

C:\Users\Ninja\AppData\Local\Temp\ipykernel_24940\2077588899.py:14: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.set_index("timestamp", inplace=True)
C:\Users\Ninja\AppData\Local\Temp\ipykernel_24940\2077588899.py:15: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.sort_index(inplace=True)


## **Task 5**

In [ ]:
log_file_paths = (find_log_files(Path("../Deliverable_2/data/")))
active_registers = (find_active_registers(Path("../Deliverable_2/data/store_config.json")))

for log_path in log_file_paths:
    if active_registers in 
    create_dataframe_from_logs(log_parser(log_path))



                    register_id     item_id  price currency   status
timestamp                                                           
2026-10-15 07:02:15      POS_01  Cappuccino   3.78      USD  SUCCESS
2026-10-15 07:03:53      POS_01    Sandwich   7.47      EUR  SUCCESS
                    register_id     item_id  price currency     status
timestamp                                                             
2026-10-15 07:02:52      POS_02  Cappuccino   3.93      EUR    SUCCESS
2026-10-15 07:05:37      POS_02    Sandwich   7.45      NOK  CANCELLED
                    register_id    item_id  price currency     status
timestamp                                                            
2026-10-15 07:02:43      POS_03   Espresso   3.03      USD    SUCCESS
2026-10-15 07:04:23      POS_03  Croissant   3.71      NOK  CANCELLED
                    register_id    item_id  price currency     status
timestamp                                                            
2026-10-15 07:01:20 